# Module 0 — Setup & Smoke Test

Welcome to **Designing and Debugging Production-Ready Multi-Agent Systems with LangGraph**.

This notebook verifies your environment is ready for the workshop.

**What this notebook does:**
1. Installs the dependencies we'll use today
2. Picks up your `OPENAI_API_KEY` and `LANGSMITH_API_KEY`
3. Runs a tiny smoke test against OpenAI
4. Prints **`Setup OK`** when everything is green

Run all cells top-to-bottom. The whole thing takes ~30 seconds.

## 1.  Install dependencies

If you're running in Colab, this cell installs everything you need. If you're running locally and have already installed the workshop's `requirements.txt`, you can skip this.

In [ ]:
# Install workshop dependencies — quiet flag suppresses noisy install logs
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    langsmith==0.1.* \
    pydantic==2.* \
    openai==1.*

print("Dependencies installed.")

## 2.  API keys

We use two keys in this workshop:

- **`OPENAI_API_KEY`** — for the actual LLM calls. **Required.**
- **`LANGSMITH_API_KEY`** — for tracing + evaluation in Modules 4 and 5. Recommended but not strictly required for Modules 1–3.

If you already have these, skip ahead to **section 2c**.  If you don't, the next two sections walk through how to create each one.

### 2a.  How to get an OpenAI API key

 > ⚠️  **You'll need to add 5 dollars in credits** to your OpenAI account before the key works for API calls. Free trial credits aren't generally offered anymore. Total spend across the whole workshop is typically under 1–2 dollars on `gpt-4o-mini`.

1. Go to **[platform.openai.com](https://platform.openai.com)** and sign in (or create an account if you don't have one).
2. Click your profile icon in the top-right corner and select **"View API keys"** — or go directly to **[platform.openai.com/api-keys](https://platform.openai.com/api-keys)**.
3. Click **"Create new secret key"**.
4. Give the key a name (optional but helpful for tracking usage), choose permissions if prompted, and click **Create**.
5. **Copy the key immediately and store it somewhere safe** — OpenAI only shows it once. If you lose it, you'll need to generate a new one.
6. Go to **Billing → Add payment method**, add a card, and load **at least $5** in credits.

> 🔒  Treat the key like a password. Don't commit it to git or paste it into client-side code. The pattern below stores it in an environment variable only — never on disk.

### 2b.  How to get a LangSmith API key

LangSmith is LangChain's hosted observability + evaluation tool. The **free tier covers everything we need** for this workshop.

1. Go to **[smith.langchain.com](https://smith.langchain.com)** and sign in (or sign up for a free account).
2. Click the **settings icon (gear)** in the lower-left sidebar, or go directly to **[smith.langchain.com/settings](https://smith.langchain.com/settings)**.
3. Select the **API Keys** tab.
4. Click **Create API Key**.
5. Choose the key type:
   - **Personal Access Token (PAT)** — tied to your user account, inherits your permissions across all workspaces. Best for workshop use.
   - **Service Key** — tied to a specific workspace, better for production/server use since it doesn't depend on a single user.
6. Give it a description (optional) and click **Create**.
7. **Copy the key right away** — LangSmith only displays it once.

> If you skip this step, Modules 1–3 still work fine; Modules 4 and 5 will run in "local mode" without the dashboard.

### 2c.  Load your keys

**Recommended path (Colab):** click the **🔑 key icon** in the left sidebar, add two secrets named exactly `OPENAI_API_KEY` and `LANGSMITH_API_KEY`, and toggle "Notebook access" ON for each. Set these **once** and every workshop notebook will pick them up automatically.

**Local Jupyter / fallback:** the cell below will prompt for each key if it isn't already in your environment.

> 🔒 **Don't commit your keys to source control.** This notebook keeps them in environment variables only — never written to disk.

In [ ]:

import os

def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")
_ensure_key("LANGSMITH_API_KEY", optional=True)
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "triage-workshop")



## 3.  Smoke test — talk to OpenAI

A two-line call to make sure the `OPENAI_API_KEY` works and the model is reachable. If this returns a reasonable answer, you're good.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
reply = llm.invoke("In one sentence: what is a multi-agent system?")
print(reply.content)

## 4.  Smoke test — LangGraph import

We don't run a graph yet — just confirm the package imports and the version is recent enough.

In [ ]:
import langgraph
from langgraph.graph import StateGraph, END

print(f"StateGraph =  {StateGraph.__name__}")
print(f"END        =  {END!r}")

## 5.  Final check

If you see `✅  Setup OK` below, you're ready for Module 1.

If you see anything else, raise a hand — we'll get it sorted before the talk starts.

In [ ]:
checks = {
    "OPENAI_API_KEY":   bool(os.environ.get("OPENAI_API_KEY")),
    "LANGSMITH_API_KEY": bool(os.environ.get("LANGSMITH_API_KEY")),
    "langgraph importable":  True,   # if we got here, it imported
    "openai reachable":      True,   # ditto
}

for name, ok in checks.items():
    mark = "✓" if ok else "✗"
    print(f"  {mark}  {name}")

print()
if all(checks.values()):
    print("✅  Setup OK  —  ready for Module 1.")
elif checks["OPENAI_API_KEY"] and checks["langgraph importable"] and checks["openai reachable"]:
    print("✅  Setup OK  (LangSmith key missing — you can set it before Module 4).")
else:
    print("⚠   Something's off. Scroll up and look for red text, or raise a hand.")

---

**Up next:** Module 1 — System Design for Multi-Agent Workflows.

You can leave this notebook open in a tab — you might come back to it to re-check the smoke test if things get weird later.